In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
BASE_DIR = Path(r"C:\Plegma_Programming\Evaluation")
HOUSE_FOLDERS = ["House01", "House05", "House07", "House10"]

FILES_TO_PLOT = {
    "coldstart_comparison.csv": "coldstart_load_curve_comparison.png",
    "with_history_comparison.csv": "with_history_load_curve_comparison.png",
}

TIME_COL = "timestamp"
EXPECTED_COLUMNS = ["actual_Wh", "rf_Wh", "xgb_Wh", "lgbm_Wh", "ensemble_Wh"]

# =========================================================
# HELPERS
# =========================================================
def load_comparison_csv(csv_path: Path) -> pd.DataFrame:
    """Load and validate a comparison CSV."""
    if not csv_path.exists():
        raise FileNotFoundError(f"File not found: {csv_path}")

    df = pd.read_csv(csv_path)

    if TIME_COL not in df.columns:
        raise ValueError(f"Missing '{TIME_COL}' column in {csv_path}")

    missing = [col for col in EXPECTED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {csv_path}: {missing}")

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.sort_values(TIME_COL).reset_index(drop=True)

    if df[TIME_COL].isna().any():
        raise ValueError(f"Invalid timestamps found in {csv_path}")

    return df


def plot_load_curve(df: pd.DataFrame, title: str, output_path: Path) -> None:
    """Create and save load curve comparison plot."""
    plt.figure(figsize=(12, 6))

    x = df["hour"] if "hour" in df.columns else range(len(df))

    plt.plot(x, df["actual_Wh"], marker="o", linewidth=2, label="Actual")
    plt.plot(x, df["rf_Wh"], marker="o", linewidth=1.8, label="RF")
    plt.plot(x, df["xgb_Wh"], marker="o", linewidth=1.8, label="XGB")
    plt.plot(x, df["lgbm_Wh"], marker="o", linewidth=1.8, label="LGBM")
    plt.plot(x, df["ensemble_Wh"], marker="o", linewidth=1.8, label="Ensemble")

    plt.title(title)
    plt.xlabel("Hour of day")
    plt.ylabel("Consumption (Wh)")
    plt.xticks(range(24))
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


# =========================================================
# MAIN
# =========================================================
for house in HOUSE_FOLDERS:
    house_dir = BASE_DIR / house

    if not house_dir.exists():
        print(f"[SKIP] Folder not found: {house_dir}")
        continue

    print(f"\nProcessing {house}...")

    for csv_name, output_name in FILES_TO_PLOT.items():
        csv_path = house_dir / csv_name
        output_path = house_dir / output_name

        if not csv_path.exists():
            print(f"  [SKIP] Missing file: {csv_name}")
            continue

        try:
            df = load_comparison_csv(csv_path)

            scenario = "Cold-start" if "coldstart" in csv_name.lower() else "With-history"
            title = f"{house} - {scenario} load curve comparison"

            plot_load_curve(df, title, output_path)
            print(f"  [OK] Saved: {output_path}")

        except Exception as e:
            print(f"  [ERROR] {csv_name}: {e}")

print("\nDone.")


Processing House01...
  [OK] Saved: C:\Plegma_Programming\Evaluation\House01\coldstart_load_curve_comparison.png
  [OK] Saved: C:\Plegma_Programming\Evaluation\House01\with_history_load_curve_comparison.png

Processing House05...
  [OK] Saved: C:\Plegma_Programming\Evaluation\House05\coldstart_load_curve_comparison.png
  [OK] Saved: C:\Plegma_Programming\Evaluation\House05\with_history_load_curve_comparison.png

Processing House07...
  [OK] Saved: C:\Plegma_Programming\Evaluation\House07\coldstart_load_curve_comparison.png
  [OK] Saved: C:\Plegma_Programming\Evaluation\House07\with_history_load_curve_comparison.png

Processing House10...
  [OK] Saved: C:\Plegma_Programming\Evaluation\House10\coldstart_load_curve_comparison.png
  [OK] Saved: C:\Plegma_Programming\Evaluation\House10\with_history_load_curve_comparison.png

Done.
